[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/vectors/v02_components.ipynb)

# Why doesn't this chain slide? Stevin's wedge

Fourteen equal balls on a loop of string hang over a wedge whose long slope is twice the short one: four balls on the
long slope, two on the short. Simon Stevin (1586) saw that the loop must balance, or it would run forever. The reason:
only the part of each ball's weight **along** its slope pulls, and the long slope's part is half the short one's.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from top to bottom. When the sliders appear, drag them and watch. Change any
number and run the cell again: nothing can break. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in this series of notebooks. Run this cell first.
import math
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
SUM, ALONG, ACROSS = "#a0c27c", "#6fb6bb", "#c9a0d3"   # the sum (resultant) · the along part · the across part

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def arrow(ax, p, q, col=CHALK, lw=2.8, ls="-"):
    """A vector arrow from point p to point q."""
    ax.annotate("", xy=q, xytext=p, arrowprops=dict(arrowstyle="-|>", color=col, lw=lw, ls=ls, mutation_scale=22))

def column(ax, vals, x, y, col=CHALK, label="", fmt="{:6.2f}"):
    """The arrow's numbers as a column in brackets, beside it: every arrow, two ways (axes coordinates)."""
    rows = "\n".join(fmt.format(v) for v in vals)
    ax.text(x, y, rows, color=col, fontsize=16, family="monospace", va="top", ha="left", transform=ax.transAxes,
            bbox=dict(boxstyle="square,pad=0.45", fc=SLATE2, ec=DIM))
    if label: ax.text(x, y + 0.03, label, color=DIM, fontsize=12, va="bottom", ha="left", transform=ax.transAxes)

## 1 · Stevin's wedge

Both slopes rise to the same top, at one height h. The balls are evenly spaced along the string, so a slope carries
balls in proportion to its length. Each ball pulls down its slope with (its weight) × sin θ, and sin θ = height ÷ length.

Drag the two slopes to make them longer or shorter (the floor stays the same, so the top moves). The pull on each side
changes, but the two pulls always stay equal: a longer slope carries more balls, and each pulls less. (Our model:
frictionless balls, one unit apart; balls are counted as fractions of a ball when a length isn't a whole number.)

In [ ]:
BASE = math.sqrt(15) + math.sqrt(3)        # the wedge's floor: slopes of 4 and 2 then meet exactly 1 unit up

def wedge(left_slope=4.0, right_slope=2.0):
    spacing = 1.0
    L1, L2, B = left_slope, right_slope, BASE
    x = (L1 ** 2 - L2 ** 2 + B ** 2) / (2 * B)                  # where the top sits above the floor
    if not (0 < x < B) or L1 ** 2 - x ** 2 <= 0:
        print("These two slopes can't meet over this floor: try other lengths."); return
    h = math.sqrt(L1 ** 2 - x ** 2)
    fig, ax = plt.subplots(figsize=(11, 5.4))
    ax.fill([0, x, B], [0, h, 0], color=SLATE2, ec=CHALK, lw=2.5)
    out = []
    for (L, foot) in ((L1, (0.0, 0.0)), (L2, (B, 0.0))):
        n = L / spacing                                             # balls on this slope (evenly spaced)
        per = h / L                                                 # each ball's pull, in ball-weights: sin = height ÷ length
        out.append((n, per, n * per))
        ux, uy = (foot[0] - x) / L, (foot[1] - h) / L               # down the slope, from the top
        nx, ny = (-uy, ux) if foot[0] == 0 else (uy, -ux)           # outward normal
        for k in range(int(n + 1e-9)):
            f = (k + 0.5) * spacing
            ax.add_patch(plt.Circle((x + ux * f + 0.12 * nx, h + uy * f + 0.12 * ny), 0.12, fc=SLATE, ec=CHALK, lw=2))
    ax.set_xlim(-3.4, B + 3.4); ax.set_ylim(-0.3, 4.3); ax.set_aspect("equal")
    ax.set_title(f"pull down the left: {out[0][2]:.2f}   ·   pull down the right: {out[1][2]:.2f}   (ball-weights)",
                 color=CORAL)
    angs = [math.degrees(math.asin(h / L)) for L in (L1, L2)]
    ax.set_xlabel(f"left slope {angs[0]:.1f}°, right slope {angs[1]:.1f}°  ·  our model: frictionless balls, evenly spaced")
    column(ax, out[0], 0.02, 0.80, ALONG, "left: balls · each · total")
    column(ax, out[1], 0.84, 0.80, ALONG, "right: balls · each · total")
    plt.show()

interact(wedge, left_slope=FloatSlider(value=4, min=4, max=5.75, step=0.25, description="left slope"),
         right_slope=FloatSlider(value=2, min=1.75, max=4, step=0.25, description="right slope"));

**Notice.** Whatever the two slopes, the totals are equal: (length ÷ spacing) × (height ÷ length) = height ÷ spacing,
the same on both sides, because both slopes share the one height. At the start, the right side rises at 30°: each
ball there pulls with half its weight, and each ball on the twice-as-long left side (about 14.5°) with a quarter.
Two halves, or four quarters.

## 2 · Split any arrow into its parts

Any arrow is a part east plus a part north: its **components**. An arrow of size r at angle θ above east has parts
r cos θ east and r sin θ north. Drag the size and the angle; the column of numbers changes with the arrow. The length
comes back from Pythagoras, and dividing by it gives the unit vector, one long, pointing the same way.

In [ ]:
def split(size=5.0, angle=53.0):
    t = math.radians(angle)
    east, north = size * math.cos(t), size * math.sin(t)
    fig, ax = plt.subplots(figsize=(11, 5.2))
    arrow(ax, (0, 0), (east, 0), ALONG, lw=3.2)
    arrow(ax, (east, 0), (east, north), ACROSS, lw=3.2)
    arrow(ax, (0, 0), (east, north), CHALK, lw=3.2)
    ax.plot([0, east / size], [0, north / size], color=CORAL, lw=5)        # the unit vector
    ax.set_xlim(-6.5, 15); ax.set_ylim(-6.5, 6.5); ax.set_aspect("equal")
    ax.set_xlabel("east"); ax.set_ylabel("north")
    ax.set_title(f"{east:.1f} i hat + {north:.1f} j hat   ·   length {math.hypot(east, north):.1f}")
    column(ax, (east, north), 0.66, 0.93, CHALK, "the arrow", "{:5.1f}")
    column(ax, (east / size, north / size), 0.83, 0.93, CORAL, "unit vector")
    plt.show()

interact(split, size=FloatSlider(value=5, min=0.5, max=6, step=0.5, description="size"),
         angle=FloatSlider(value=53, min=-180, max=180, step=1, description="angle (°)"));

## 3 · A box on a ramp

The weight W = mg points straight down. Split it into a part along the ramp, W sin θ, and a part into it, W cos θ.
The angle θ sits next to the part into the ramp, so that part gets cos. Flatten the ramp to 0° to check: nothing
pulls along a flat floor.

In [ ]:
def ramp(slope=30.0, mass=10.0):
    g = 9.8
    W = mass * g
    t = math.radians(slope)
    along, into = W * math.sin(t), W * math.cos(t)
    fig, ax = plt.subplots(figsize=(10, 5.2))
    L = 6
    ax.fill([0, L * math.cos(t), L * math.cos(t)], [0, 0, L * math.sin(t)], color=SLATE2, ec=CHALK, lw=2.5)
    p = (0.55 * L * math.cos(t), 0.55 * L * math.sin(t))
    s = 3.0 / W                                                      # arrow scale: the weight is 3 units long
    down = (p[0], p[1] - W * s)
    a_vec = (-along * s * math.cos(t), -along * s * math.sin(t))
    i_vec = (into * s * math.sin(t), -into * s * math.cos(t))
    arrow(ax, p, down, CHALK, lw=3.2)
    arrow(ax, p, (p[0] + a_vec[0], p[1] + a_vec[1]), ALONG, lw=3.2)
    arrow(ax, (p[0] + a_vec[0], p[1] + a_vec[1]), down, ACROSS, lw=3.2)
    ax.set_xlim(-1.5, 7.5); ax.set_ylim(-2.5, 4.2); ax.set_aspect("equal")
    ax.set_title(f"W = {W:.1f} N   ·   along {along:.2f} N (sin)   ·   into {into:.2f} N (cos)")
    column(ax, (along, into), 0.80, 0.93, CHALK, "along · into (N)")
    plt.show()

interact(ramp, slope=FloatSlider(value=30, min=0, max=80, step=1, description="ramp (°)"),
         mass=FloatSlider(value=10, min=1, max=20, step=1, description="mass (kg)"));

**Try it.** The challenge from the video: find the ramp where the part **into** the ramp is twice the part **along**
it. Hint: along ÷ into = sin θ ÷ cos θ.

## Sources
Simon Stevin, *De Beghinselen der Weeghconst* (Leiden, 1586), the clootcrans; MacTutor, "Simon Stevin"; NCERT Physics
class XI ch. 3 (resolution of vectors) and Mathematics class XII ch. 10 (components, unit vector), 2026–27 reprints.
The 30° side is our own choice: Stevin's figure fixes only the 2 : 1 lengths. Retrieved 2026-10-26.